# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManjusreeValluri/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use a **Decision Tree Classifier** for this modeling task.

A Decision Tree fits this lane because it can capture simple non-linear relationships between content features and the target, while remaining easy to interpret. It also allows us to see which feature conditions are used to make predictions.

I will keep the tree shallow to avoid unnecessary complexity and compare its performance against the Week-4 hand-rule baseline using the same evaluation metric.

The goal is not to build the most complex model, but to check whether the model provides a useful improvement over the existing baseline.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)

Shape: (30000, 44)


In [32]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

print("Method: Decision Tree Classifier")
print("Maximum depth:", model.max_depth)

Method: Decision Tree Classifier
Maximum depth: 2


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **grouped split by client** so that pages from the same client do not appear in both the training and validation sets.

This is a more honest evaluation because the model should be tested on clients it did not see during training. It reduces the risk of data leakage and gives a better estimate of how the model may perform on new clients.

I will use the same split design and evaluation setup as the Week-4 baseline wherever possible so the comparison is fair.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Target: 1 = CTR_FIX, 0 = REFRESH_SUPPORT
df["baseline_target"] = (
    df["position_tier"]
    .isin(["deep", "page_3_5", "striking"])
    .astype(int)
)

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].copy()
y = df["baseline_target"]

# Fill missing word_count
X["word_count"] = X["word_count"].fillna(X["word_count"].median())

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values:", X.isna().sum().sum())

Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
X shape: (30000, 6)
y shape: (30000,)
Missing values: 0


In [34]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_test))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Validation clients:", df.iloc[test_idx]["client_id"].nunique())

overlap = len(
    set(df.iloc[train_idx]["client_id"])
    & set(df.iloc[test_idx]["client_id"])
)

print("Client overlap:", overlap)

Training rows: 23837
Validation rows: 6163
Training clients: 25
Validation clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [35]:
import os

print(os.listdir("/content"))

['.config', 'flyrank-ml-internship', 'sample_data']


In [36]:
!git clone https://github.com/ManjusreeValluri/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [37]:
import os

print(os.listdir("/content/flyrank-ml-internship"))

['docs', 'outputs', 'SETUP.md', 'CLAUDE.md', 'LICENSE', 'submission', 'requirements.txt', 'skills', 'AGENTS.md', 'notebooks', '.git', 'GUIDE.md', 'scripts', 'work', '.gitignore', '.github', 'README.md', 'data', 'DATA_USE.md']


In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [39]:
print(df["trend_direction"].value_counts(dropna=False))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [40]:
import os

print(os.listdir("/content/flyrank-ml-internship/work"))

['capstone_report_template.md', 'notebooks', 'README.md']


In [41]:
import os

print(os.listdir("/content/flyrank-ml-internship/work/notebooks"))

['w06_validation_audit.ipynb', 'w05_model.ipynb', 'w04_baseline_score (1).ipynb', 'w02_ml_task_framing.ipynb', 'w03_data_contract.ipynb', 'w07_action_playbook.ipynb', 'w04_signal_audit.ipynb', 'w01_research_question.ipynb', 'capstone.ipynb', 'w04_baseline_score.ipynb', 'w03_feature_leakage_check.ipynb']


In [42]:
print(os.listdir("/content/flyrank-ml-internship/work/notebooks"))

['w06_validation_audit.ipynb', 'w05_model.ipynb', 'w04_baseline_score (1).ipynb', 'w02_ml_task_framing.ipynb', 'w03_data_contract.ipynb', 'w07_action_playbook.ipynb', 'w04_signal_audit.ipynb', 'w01_research_question.ipynb', 'capstone.ipynb', 'w04_baseline_score.ipynb', 'w03_feature_leakage_check.ipynb']


In [43]:
import numpy as np
import os

# Start with a copy of the dataset
queue = df.copy()

# Score based mainly on search position
queue["score"] = 0.0

# Higher priority for pages with weaker search positions
queue.loc[queue["position_tier"] == "deep", "score"] += 4
queue.loc[queue["position_tier"] == "page_3_5", "score"] += 3
queue.loc[queue["position_tier"] == "striking", "score"] += 2
queue.loc[queue["position_tier"] == "page_1", "score"] += 1
queue.loc[queue["position_tier"] == "top_3", "score"] += 0

# Freshness is supporting context only
queue.loc[
    queue["freshness_tier"].isin(["91-180", "181+"]),
    "score"
] += 1

# Reason code
queue["reason_code"] = np.where(
    queue["position_tier"].isin(
        ["deep", "page_3_5", "striking"]
    ),
    "CTR_FIX",
    "REFRESH_SUPPORT"
)

# Action
queue["action"] = np.where(
    queue["reason_code"] == "CTR_FIX",
    "Improve CTR",
    "Review freshness"
)

# Rank
queue = queue.sort_values(
    ["score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

baseline_queue = queue[
    ["rank", "content_id", "score", "reason_code", "action"]
].copy()

print("Baseline recreated successfully.")
print("Rows:", len(baseline_queue))
print("\nTop 10:")
display(baseline_queue.head(10))

Baseline recreated successfully.
Rows: 30000

Top 10:


,rank,content_id,score,reason_code,action
0,1,content_00bd50f6286d,5.0,CTR_FIX,Improve CTR
1,2,content_01e6f180a032,5.0,CTR_FIX,Improve CTR
2,3,content_0239cfc771ee,5.0,CTR_FIX,Improve CTR
3,4,content_026300c5cc1e,5.0,CTR_FIX,Improve CTR
4,5,content_0334348fc2d9,5.0,CTR_FIX,Improve CTR
5,6,content_0371fc7cdb0a,5.0,CTR_FIX,Improve CTR
6,7,content_03b7a0f49f84,5.0,CTR_FIX,Improve CTR
7,8,content_04e97796ac4d,5.0,CTR_FIX,Improve CTR
8,9,content_0527b708fdec,5.0,CTR_FIX,Improve CTR
9,10,content_07445a84d693,5.0,CTR_FIX,Improve CTR


In [44]:
# Create the target used for the model
# 1 = baseline recommends CTR improvement
# 0 = otherwise

df["baseline_target"] = (
    df["position_tier"].isin(
        ["deep", "page_3_5", "striking"]
    ).astype(int)
)

print(df["baseline_target"].value_counts())
print("\nTarget meaning:")
print("1 = CTR_FIX / Improve CTR")
print("0 = REFRESH_SUPPORT / Review freshness")

baseline_target
1    15865
0    14135
Name: count, dtype: int64

Target meaning:
1 = CTR_FIX / Improve CTR
0 = REFRESH_SUPPORT / Review freshness


In [45]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features]
y = df["baseline_target"]

print("Features:")
print(features)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values:")
print(X.isna().sum())

Features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

X shape: (30000, 6)
y shape: (30000,)

Missing values:
content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699
dtype: int64


In [46]:
# Fill missing word_count values with the median
X = X.copy()

X["word_count"] = X["word_count"].fillna(X["word_count"].median())

print("Missing values after filling:")
print(X.isna().sum())

Missing values after filling:
content_age_days          0
days_since_last_update    0
impressions_90d           0
avg_position              0
ctr                       0
word_count                0
dtype: int64


In [47]:
from sklearn.model_selection import GroupShuffleSplit

# Use client_id as the grouping variable
groups = df["client_id"]

# 80% training, 20% validation
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_test))

print("\nTraining clients:", df.iloc[train_idx]["client_id"].nunique())
print("Validation clients:", df.iloc[test_idx]["client_id"].nunique())

print("\nClients overlap:",
      len(
          set(df.iloc[train_idx]["client_id"])
          & set(df.iloc[test_idx]["client_id"])
      ))

Training rows: 23837
Validation rows: 6163

Training clients: 25
Validation clients: 7

Clients overlap: 0


In [48]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [49]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Predict on validation clients
y_pred = model.predict(X_test)

# Calculate validation metrics
model_accuracy = accuracy_score(y_test, y_pred)
model_precision = precision_score(y_test, y_pred, zero_division=0)
model_recall = recall_score(y_test, y_pred, zero_division=0)
model_f1 = f1_score(y_test, y_pred, zero_division=0)

print("Decision Tree validation results")
print("--------------------------------")
print(f"Accuracy : {model_accuracy:.3f}")
print(f"Precision: {model_precision:.3f}")
print(f"Recall   : {model_recall:.3f}")
print(f"F1 Score : {model_f1:.3f}")

Decision Tree validation results
--------------------------------
Accuracy : 0.999
Precision: 1.000
Recall   : 0.999
F1 Score : 0.999


In [50]:
# Get model probability for the positive class (CTR_FIX)
model_scores = model.predict_proba(X_test)[:, 1]

# True labels for validation set
y_test_array = y_test.to_numpy()

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores, kind="stable")[:k]
    return y_true[order].mean()

model_p20 = precision_at_k(model_scores, y_test_array, 20)
model_p50 = precision_at_k(model_scores, y_test_array, 50)

print("Decision Tree ranking results")
print("-----------------------------")
print(f"Precision@20: {model_p20:.3f}")
print(f"Precision@50: {model_p50:.3f}")

Decision Tree ranking results
-----------------------------
Precision@20: 1.000
Precision@50: 1.000


In [51]:
comparison = pd.DataFrame({
    "Model": ["Week-4 Baseline", "Decision Tree"],
    "Precision@20": [baseline_p20, model_p20],
    "Precision@50": [baseline_p50, model_p50]
})

display(comparison)

,Model,Precision@20,Precision@50
0,Week-4 Baseline,1.0,1.0
1,Decision Tree,1.0,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Decision Tree achieved the same measured ranking performance as the Week-4 baseline on the client-held-out validation set. Both methods achieved Precision@20 of 1.000 and Precision@50 of 1.000.

The feature importance results show that `avg_position` was the main feature used by the Decision Tree, with an importance of approximately 1.00. The other selected features had negligible importance.

This is expected because the target was derived from the Week-4 baseline rule, which is based mainly on search-position information. Therefore, the model appears to have learned the existing baseline decision pattern rather than discovering an independently validated improvement.

The observed results support the Decision Tree as an interpretable way to reproduce the baseline decisions, but they do not show that the model will improve actual CTR. The result should therefore be treated as decision-support evidence rather than proof of a production improvement.


In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

,Feature,Importance
3,avg_position,1.000000e+00
5,word_count,3.866281e-13
1,days_since_last_update,0.000000e+00
0,content_age_days,0.000000e+00
2,impressions_90d,0.000000e+00
4,ctr,0.000000e+00


In [53]:
# Recalculate the Week-4 baseline on the same validation set

baseline_test = baseline_queue[
    baseline_queue["content_id"].isin(df.iloc[test_idx]["content_id"])
].copy()

baseline_p20 = (baseline_test["reason_code"].head(20) == "CTR_FIX").mean()
baseline_p50 = (baseline_test["reason_code"].head(50) == "CTR_FIX").mean()

print("Baseline Precision@20:", round(baseline_p20, 3))
print("Baseline Precision@50:", round(baseline_p50, 3))

Baseline Precision@20: 1.0
Baseline Precision@50: 1.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.